# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset and view its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their components using their `@id` values.

In [ ]:
# List all record sets by `@id` and print available fields and columns
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for f in rs.fields:
            print(f"    - @id: {f.id}, name: {f.name}, dataType: {getattr(f, 'data_type', 'N/A')}")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for c in rs.columns:
            print(f"    - @id: {c.id}, name: {c.name}, dataType: {getattr(c, 'data_type', 'N/A')}")
    print()

Below we demonstrate how to iterate through all records for a record set by its `@id`.

In [ ]:
# Pick the first record set for demonstration
if len(record_sets) > 0:
    demo_record_set = record_sets[0].id
    print(f"\nPreviewing 3 records from record set @id: {demo_record_set}")
    for i, record in enumerate(dataset.records(record_set=demo_record_set)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for subsequent analysis. All record sets are referenced by their `@id`.

In [ ]:
# Build a list of all record set IDs
record_set_ids = [rs.id for rs in record_sets]

# Load each record set into a DataFrame by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Sometimes record may be {}, skip empty sets
    if records and isinstance(records[0], dict):
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}")
    else:
        print(f"Skipped (empty or invalid): {record_set_id}")

# Demo: Print columns of the first populated DataFrame
first_populated_id = None
for k, v in dataframes.items():
    if not v.empty:
        first_populated_id = k
        break
if first_populated_id:
    print(f"\nColumns in record set @id={first_populated_id}: {dataframes[first_populated_id].columns.tolist()}")
    display(dataframes[first_populated_id].head())
else:
    print("No dataframes loaded with records.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, and grouping by key fields. This example assumes numeric and groupable columns exist in the first populated record set.

In [ ]:
# Select a record set with at least one numeric field for analysis
if first_populated_id is not None:
    df = dataframes[first_populated_id]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use first numeric field
        print(f"Using numeric field: {numeric_field_id}")
    else:
        numeric_field_id = None
        print("No numeric fields found for numeric operations.\n")
    
    # Filtering example
    if numeric_field_id:
        # Choose threshold as the median value
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by a potential categorical field
        groupable_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = None
        if groupable_cols:
            # Heuristically select a group field that is not unique for every row
            for col in groupable_cols:
                if df[col].nunique() < (0.5 * len(df)):
                    group_field_id = col
                    break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_"+numeric_field_id)
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions and relationships, using fields referenced by `@id`.


In [ ]:
# Example: Histogram of the numeric field; boxplot by group (if applicable)
if first_populated_id is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_field_id].dropna(), bins=10, color="skyblue", edgecolor="black")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and perform basic preprocessing and visualization of a dataset defined by a Croissant schema. 

Key steps included:
- Loading Croissant metadata and tabular data via entity `@id`s
- Programmatic overview of record set, fields, and column `@id`s
- DataFrame creation for selected tables using these identifiers
- Example EDA: filtering, normalization, grouping, and visualization

For further analysis (statistical testing, machine learning, or deeper domain-specific insights), continue leveraging the `@id`-driven column access pattern as shown.